# UAV adversarial robustness — Colab runner

Runtime → Change runtime type → **GPU** (T4 is enough; A100 is about 4× faster).

Every stage caches its outputs on Google Drive, so after a disconnect re-run the cells from the top and they resume where they stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Get the code
Option A: push this repo to GitHub and set `REPO_URL`. Option B: upload the repo folder to `MyDrive/uavadv/code`.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/uav-adv-robustness.git'
import os
if REPO_URL:
    !git clone -q $REPO_URL /content/uavadv-code 2>/dev/null || git -C /content/uavadv-code pull -q
    CODE = '/content/uavadv-code'
else:
    CODE = '/content/drive/MyDrive/uavadv/code'
%cd $CODE
!pip -q install -r requirements.txt

## Phase 1 — data and clean baseline
Downloads about 1.8 GB and converts it to YOLO format on the local disk (about 5 minutes). The full 100-epoch fine-tune of yolo11n takes about 3–4 h on a T4; for a first pass use `--set train.epochs=30`.

In [ ]:
CFG = '--config configs/colab.yaml'
!python scripts/prepare_data.py $CFG

In [ ]:
!python scripts/train.py $CFG --set train.epochs=30

Sanity check: clean test mAP plus a manual look at predictions (proposal Sec. IV-A).

In [ ]:
from ultralytics import YOLO
import glob
m = YOLO('/content/drive/MyDrive/uavadv/runs/baseline/weights/best.pt')
imgs = sorted(glob.glob('/content/drive/MyDrive/uavadv/eval_sets/val_640/clean/images/*.png'))[:4]
for r in m.predict(imgs, conf=0.25, verbose=False):
    r.show()

## Phase 3a — calibrate the defense on **val**

In [ ]:
!python scripts/calibrate_defense.py $CFG
!cat /content/drive/MyDrive/uavadv/results/chosen_defense.json | head -5

## Phase 2 + 3 — all Table I conditions on **test**
FGSM/PGD at ε ∈ {1,2,4,8}/255, no defense vs the chosen defense, plus the adaptive PGD attack. For the final run use `--set attack.seeds=[0,1,2]`.

In [ ]:
!python scripts/run_all.py $CFG

## Optional: adversarial training (stronger defense baseline)

In [ ]:
!python scripts/train.py $CFG --set adv_train.enabled=true
!python scripts/run_all.py $CFG --weights /content/drive/MyDrive/uavadv/runs/advtrain/weights/best.pt --tag advtrain

## Figures and tables

In [ ]:
!python scripts/plot_results.py $CFG
from IPython.display import Image, display
for f in ['map_vs_eps','recall_by_class','recall_by_scale','latency']:
    display(Image(f'/content/drive/MyDrive/uavadv/results/figures/{f}.png'))